In [1]:
%pip install ucimlrepo --quiet


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

ds = fetch_ucirepo(id=601)   # 601 = AI4I 2020 Predictive Maintenance
df_raw = pd.concat([ds.data.features, ds.data.targets], axis=1)

print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head()

Shape: (10000, 12)
Columns: ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
df_raw = pd.read_csv("../data/ai4i2020.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../data/ai4i2020.csv'

In [4]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

ds = fetch_ucirepo(id=601)
df_raw = pd.concat([ds.data.features, ds.data.targets], axis=1)

print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head()

Shape: (10000, 12)
Columns: ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [5]:
import os
print("Current folder:", os.getcwd())
print("Files in ../data:", os.listdir("../data"))


Current folder: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-umme-kalsoom-lab1/code/Users/kalsoomume779/azureml-knn-maintenance/notebooks
Files in ../data: ['.amlignore', '.amlignore.amltmp', '.gitkeep', 'ai4i2020.csv']


In [6]:
df_raw = pd.read_csv("../data/ai4i2020.csv")
print("Shape:", df_raw.shape)
df_raw.head()

Shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [7]:
df = df_raw.copy()
df.columns = [c.split(" [")[0].strip() for c in df.columns]

rename_map = {
    "Type": "type",
    "Air temperature": "air_temp_k",
    "Process temperature": "process_temp_k",
    "Rotational speed": "rpm",
    "Torque": "torque_nm",
    "Tool wear": "tool_wear_min",
    "Machine failure": "machine_failure",
}
df = df.rename(columns=rename_map)[list(rename_map.values())]
df.head()

,type,air_temp_k,process_temp_k,rpm,torque_nm,tool_wear_min,machine_failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [8]:
print("Missing values per column:")
print(df.isna().sum())
print("\nData types:")
print(df.dtypes)
print("\nTarget balance (share of rows):")
print(df["machine_failure"].value_counts(normalize=True).round(4))


Missing values per column:
type               0
air_temp_k         0
process_temp_k     0
rpm                0
torque_nm          0
tool_wear_min      0
machine_failure    0
dtype: int64

Data types:
type                object
air_temp_k         float64
process_temp_k     float64
rpm                  int64
torque_nm          float64
tool_wear_min        int64
machine_failure      int64
dtype: object

Target balance (share of rows):
0    0.9661
1    0.0339
Name: machine_failure, dtype: float64


In [9]:
df.to_csv("../data/ai4i_clean.csv", index=False)
print("Saved", df.shape, "to data/ai4i_clean.csv")

Saved (10000, 7) to data/ai4i_clean.csv


In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<yahan-apni-subscription-id>"
RESOURCE_GROUP = "ML_KNN_PROJECT_BY_UMM-E-KALSOOM"
WORKSPACE = "rg-mlab1-umme-kalsoom"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID,
                     RESOURCE_GROUP, WORKSPACE)
print("Connected to:", ml_client.workspace_name)

Connected to: rg-mlab1-umme-kalsoom


In [2]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

csv_asset = Data(
    name="ai4i-maintenance-csv",
    version="1",
    type=AssetTypes.URI_FILE,
    path="../data/ai4i_clean.csv",
    description="AI4I 2020 predictive maintenance, cleaned: 6 features + machine_failure",
)
csv_asset = ml_client.data.create_or_update(csv_asset)
print("Registered:", csv_asset.name, "version", csv_asset.version)

HttpResponseError: (InvalidSubscriptionId) The provided subscription identifier '<yahan-apni-subscription-id>' is malformed or invalid.
Code: InvalidSubscriptionId
Message: The provided subscription identifier '<yahan-apni-subscription-id>' is malformed or invalid.

In [3]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "1e635088-8857-401c-ad7a-bb6863d85fac"
RESOURCE_GROUP = "ML_KNN_PROJECT_BY_UMM-E-KALSOOM"
WORKSPACE = "rg-mlab1-umme-kalsoom"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID,
                     RESOURCE_GROUP, WORKSPACE)
print("Connected to:", ml_client.workspace_name)

Connected to: rg-mlab1-umme-kalsoom


In [4]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

csv_asset = Data(
    name="ai4i-maintenance-csv",
    version="1",
    type=AssetTypes.URI_FILE,
    path="../data/ai4i_clean.csv",
    description="AI4I 2020 predictive maintenance, cleaned: 6 features + machine_failure",
)
csv_asset = ml_client.data.create_or_update(csv_asset)
print("Registered:", csv_asset.name, "version", csv_asset.version)

Uploading ai4i_clean.csv (< 1 MB): 100%|██████████| 295k/295k [00:00<00:00, 11.8MB/s]




Registered: ai4i-maintenance-csv version 1


In [5]:
import os
print(os.listdir("../data"))

['.amlignore', '.amlignore.amltmp', '.gitkeep', 'ai4i2020.csv', 'ai4i_clean.csv']


In [6]:
cd ~/cloudfiles/code/Users/*/azureml-knn-maintenance
git add data/ai4i_clean.csv notebooks/00_prepare_data.ipynb
git commit -m "Part 4: clean AI4I dataset and data prep notebook"
git push

SyntaxError: invalid decimal literal (1367827645.py, line 2)

In [8]:
import os, shutil
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

# 1. Ek folder banayein jismein CSV aur MLTable file dono hon
folder = "../data/mltable_ai4i"
os.makedirs(folder, exist_ok=True)
shutil.copy("../data/ai4i_clean.csv", f"{folder}/ai4i_clean.csv")

# 2. MLTable file (batati hai ke CSV kahan hai aur kaise parhni hai)
with open(f"{folder}/MLTable", "w") as f:
    f.write("""paths:
  - file: ./ai4i_clean.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: 'utf8'
      header: all_files_same_headers
""")

# 3. Workspace se connect aur asset register (version 2)
ml_client = MLClient(DefaultAzureCredential(),
                     "1e635088-8857-401c-ad7a-bb6863d85fac",
                     "ML_KNN_PROJECT_BY_UMM-E-KALSOOM",
                     "rg-mlab1-umme-kalsoom")

table_asset = Data(
    name="ai4i-maintenance-table",
    version="2",
    type=AssetTypes.MLTABLE,
    path=folder,
    description="AI4I 2020 cleaned, mltable for AutoML",
)
table_asset = ml_client.data.create_or_update(table_asset)
print("Registered:", table_asset.name, "version", table_asset.version)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Uploading mltable_ai4i (0.3 MBs): 100%|██████████| 295096/295096 [00:00<00:00, 5012645.12it/s]




Registered: ai4i-maintenance-table version 2
